In [38]:
import polars as pl
import os

from pathlib import Path

In [39]:
DONOR = 'donor_4'

In [40]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined' / DONOR

In [41]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [42]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",39,71.0,0.78,71.5,0.8058,0.6984,0.761,0.7283,0.888
"""LogisticRegression""",1011,"""H3K4me3""",null,69.8,0.7644,71.8,0.7992,0.7523,0.6534,0.6994,0.762
"""RandomForest""",1011,"""H3K4me3""",null,70.4,0.7813,72.1,0.8057,0.7528,0.6614,0.7041,0.767
"""SVM_Linear""",1011,"""H3K4me3""",null,69.9,0.7636,71.7,0.801,0.7506,0.6534,0.6986,0.76
"""DirectRanker""",123,"""H3K4me3""",60,74.1,0.83,71.6,0.8145,0.6891,0.7703,0.7274,0.892
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.9,0.8033,71.4,0.755,0.7315,0.6502,0.6885,0.741
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",80,72.4,0.81,74.9,0.8361,0.7386,0.7753,0.7565,0.927
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,69.7,0.7381,71.1,0.7686,0.7477,0.6421,0.6909,0.735


In [43]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [44]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",73.68,2.7662,0.8187,0.0219,74.94,1.3465,0.8301,0.0154
"""H3K27ac""","""SVM_Linear""",72.24,2.5822,0.788,0.0238,73.22,1.4025,0.8022,0.016
"""H3K27ac""","""LogisticRegression""",72.48,2.7698,0.7883,0.025,73.2,1.3096,0.802,0.0169
"""H3K27ac""","""DirectRanker""",72.36,2.0182,0.814,0.0288,72.88,0.9884,0.8198,0.0112
"""H3K27ac-H3K27me3""","""RandomForest""",73.66,2.7144,0.8201,0.0212,75.02,1.6769,0.8315,0.0153
…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",70.72,2.543,0.7633,0.0244,71.26,0.9182,0.7704,0.0168
"""H3K9me3-H3K27me3""","""RandomForest""",53.62,1.3442,0.5392,0.0088,52.68,1.1925,0.546,0.0074
"""H3K9me3-H3K27me3""","""DirectRanker""",51.12,1.0941,0.55,0.0071,52.6,0.8944,0.554,0.0188


In [45]:
summary_df.write_csv(OUTPUT_PATH/ f"{DONOR}.csv", include_header=True)